# 01 Data Exploration

This notebook loads the UCI Online Retail dataset, checks raw data quality, cleans transactions, and creates the requested exploratory charts. The goal is to understand the business data before modeling.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import CLEANED_TRANSACTIONS_PATH, CUSTOMER_FEATURES_PATH
from src.data_loader import download_dataset, load_raw_transactions, summarize_raw_transactions
from src.feature_engineering import build_customer_features
from src.preprocessing import clean_transactions, print_cleaning_report
from src.reporting import create_eda_figures

## Load Raw Data

We download the real UCI dataset if needed and inspect shape, columns, missing values, duplicate rows, unique customers, date range, and cancelled transactions.

In [ ]:
raw_path = download_dataset()
raw_df = load_raw_transactions(raw_path)
raw_summary = summarize_raw_transactions(raw_df)

## Clean Transactions

Cleaning removes missing customers, cancelled invoices, invalid quantities/prices, and duplicates. Revenue is created as `Quantity * UnitPrice`.

In [ ]:
clean_df, cleaning_stats = clean_transactions(raw_df)
print_cleaning_report(cleaning_stats)
clean_df.to_csv(CLEANED_TRANSACTIONS_PATH, index=False)
clean_df.head()

## Business Questions

These checks answer the first interview-style business questions from the workflow.

In [ ]:
orders = clean_df['InvoiceNo'].nunique()
customers = clean_df['CustomerID'].nunique()
total_revenue = clean_df['Revenue'].sum()
average_order_value = clean_df.groupby('InvoiceNo')['Revenue'].sum().mean()

print(f'Customers: {customers:,}')
print(f'Transactions/orders: {orders:,}')
print(f'Total revenue: {total_revenue:,.2f}')
print(f'Average order value: {average_order_value:,.2f}')

clean_df.groupby('Country')['CustomerID'].nunique().sort_values(ascending=False).head(10)

In [ ]:
customer_features, metadata = build_customer_features(clean_df)
customer_features.to_csv(CUSTOMER_FEATURES_PATH, index=False)
create_eda_figures(clean_df, customer_features)
metadata